# ♟️ Chess Numba — NMP 離散參數驗證錦標賽 (Colab Runner)

本 Notebook 用於在 Google Colab 雲端高並行環境下，針對 **NMP 離散小整數參數** 進行成對換先（Color-swapped Pairs）LTC 錦標賽對戰驗證：

### 🎯 本輪對抗設定 (Round 1: Verification Depth)
* **Engine 1 (New / Candidate)**: `chess_engine/classical` (`NMP_VERIFICATION_DEPTH = 14`)
* **Engine 2 (Old / Baseline)**: `chess_engine/classical_old` (`NMP_VERIFICATION_DEPTH = 16`，SF19 原生基準)
* **時控規範**: 300,000 nodes（長時控深度視野，平均 depth 18~20）
* **對戰規模**: 400 局（200 對換先開局配對），64MB 轉置表，開局庫 `data/openings.epd`
* **檢驗核心**: 檢驗在 $d=14\sim 15$ 層啟動輕量驗證搜尋（Verification Search），能否在幾乎不增加節點開銷的前提下，為複雜中局提供更強的防 Zugzwang / 防棄子防護網。


In [ ]:
# 1. 檢視 Colab 雲端硬體與 CPU 核心數
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!free -h
!python3 --version


In [ ]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = 'feat-initial-bitboard-setup'

if not os.path.exists('/content/chess_numba'):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git fetch origin {branch}
!git checkout -f {branch}
!git reset --hard origin/{branch}
!git clean -fd
!pip install -q numba chess numpy matplotlib scipy


In [ ]:
# 3. 嚴格檢驗新舊版本配置與離散參數差異
import sys
from pathlib import Path

# 1) 清理 Python 模組記憶體快取
for m in list(sys.modules.keys()):
    if 'chess_engine' in m or 'tune_search' in m:
        del sys.modules[m]

# 2) 清除過期 Numba 編譯快取並同步快照
!python -m tuner.sync_constants_snapshot

# 3) 檢驗新舊引擎差異（必須僅有 NMP_VERIFICATION_DEPTH 不同）
!python tools/sync_classical_old.py --diff

import chess_engine.classical.constants as C_new
import chess_engine.classical_old.constants as C_old

print("\n" + "="*50)
print(f"Candidate (New) NMP_VERIFICATION_DEPTH : {C_new.NMP_VERIFICATION_DEPTH}")
print(f"Baseline  (Old) NMP_VERIFICATION_DEPTH : {C_old.NMP_VERIFICATION_DEPTH}")
print(f"NMP_BASE            : New={C_new.NMP_BASE}, Old={C_old.NMP_BASE}")
print(f"NMP_EVAL_MARGIN_DIV : New={C_new.NMP_EVAL_MARGIN_DIV}, Old={C_old.NMP_EVAL_MARGIN_DIV}")
print(f"NMP_DEPTH_COEF      : New={C_new.NMP_DEPTH_COEF}, Old={C_old.NMP_DEPTH_COEF}")
print(f"NMP_R_BASE          : New={C_new.NMP_R_BASE}, Old={C_old.NMP_R_BASE}")
print("="*50 + "\n")

assert C_new.NMP_VERIFICATION_DEPTH == 14, f"Expected New=14, got {C_new.NMP_VERIFICATION_DEPTH}"
assert C_old.NMP_VERIFICATION_DEPTH == 16, f"Expected Old=16, got {C_old.NMP_VERIFICATION_DEPTH}"
print("🎉 參數配置 100% 正確，準備開戰！")


In [ ]:
# 4. 執行對抗錦標賽（300k nodes, 400 局, concurrency 7, TT 64MB）
%cd /content/chess_numba
!python tools/tournament.py --games 400 --nodes 300000 --concurrency 7 --tt-mb 64


In [ ]:
# 5. 產出詳細統計數據與搜尋指標分析報告
%cd /content/chess_numba
!python tournament_analysis/統計數據.py
!python tournament_analysis/parse_search_stats.py

report_path = Path("tournament_analysis/game_stats_report.txt")
if report_path.exists():
    print("\n" + "="*60)
    print(report_path.read_text(encoding='utf-8'))


In [ ]:
# 6. 視覺化關鍵圖表並打包下載
import os
from IPython.display import Image, display

charts = [
    "tournament_analysis/cumulative_score.png",
    "tournament_analysis/elo_ci.png",
    "tournament_analysis/depth_per_move.png",
    "tournament_analysis/same_depth_node_ratio.png",
]
for chart in charts:
    if os.path.exists(chart):
        print(f"\n📊 圖表: {chart}")
        display(Image(filename=chart))

# 打包所有報告、圖表與 PGN 棋譜
!zip -q tournament_results_nmp_verify14.zip tournament_analysis/*.png tournament_analysis/*.txt tournament_analysis/tournament_results.pgn
print("\n📦 已生成壓縮包: tournament_results_nmp_verify14.zip")

try:
    from google.colab import files
    files.download('tournament_results_nmp_verify14.zip')
    print("⬇️ 已觸發下載！")
except ImportError:
    pass
